# UNO Vision — IAPR 2026 Final Project

Pipeline d'analyse d'images de parties UNO. Pour chaque image (4000×2662, fond blanc ou bruité), on prédit :
- la **carte centrale** sur la table,
- le **joueur actif** (parmi p1/p2/p3/p4, indiqué par un jeton),
- les **cartes en main** de chaque joueur.

Le score Kaggle est `0.1·CenterAcc + 0.1·ActiveAcc + 0.8·F1`. La baseline DL est à 0.647.

## Approche choisie : pipeline hybride classique + DL

1. **Détection** des cartes par segmentation HSV (saturation floue + détection des ovales blancs enclosed pour fond bruité)
2. **Classification** de chaque carte par un CNN ResNet-light (~11.2M params) entraîné from scratch sur un dataset synthétique généré depuis les `reference_images` officielles
3. **Détection du jeton actif** (noir rectangulaire sur fond blanc, jaune rond sur fond bruité)
4. **Assignation géométrique** : chaque carte → joueur le plus proche (p1/p2/p3/p4) ou center

## Contraintes du règlement
- ❌ Pas de datasets externes — on n'utilise que `reference_images/` officielles
- ❌ Pas de modèles pré-entraînés — entraînement from scratch
- ❌ ≤ 12M paramètres — notre modèle fait 11.2M
- ❌ Test images uniquement pour l'inférence finale


In [ ]:
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.config import CARD_CLASSES, NUM_CLASSES, REFERENCE_DIR, TRAIN_DIR, TRAIN_CSV

print(f'Number of card classes : {NUM_CLASSES}')
print(f'Number of train images : {len(list(TRAIN_DIR.glob("*.jpg")))}')

## 1. Le dataset

On dispose de :
- 81 images **train** annotées (cartes par joueur + carte centrale + joueur actif)
- 159 images **test** non annotées
- 4 **reference_images** : photos studio des 54 cartes UNO sur fond blanc, organisées en grilles

Les 54 classes : 4 couleurs × (10 chiffres 0–9 + 3 actions skip/reverse/draw_2) + 2 wilds (wild, draw_4).

Difficulté principale : **81 images train est très peu pour 54 classes**. Un CNN entraîné directement sur ces images va overfitter immédiatement.

→ La stratégie est de partir des `reference_images` et de générer un **dataset synthétique** par augmentation lourde.

In [ ]:
# Visualisation d'une scène train annotée
df_train = pd.read_csv(TRAIN_CSV)
print(df_train.head().to_string(index=False))

## 2. Extraction des templates depuis les reference_images

Les 4 `reference_images` contiennent toutes les cartes du jeu, photographiées dans des grilles régulières. On en extrait les 54 templates propres en 200×300 par :
1. **Segmentation** : HSV saturation floue → blob plein par carte
2. **Split géométrique** des blobs trop grands (cartes adjacentes mergées par le flou)
3. **Matching par couleur** : chaque carte détectée → label selon sa couleur dominante + position dans le layout connu de l'image
4. **Fallbacks hardcodés** pour 3 cartes coincées dans un L-shape merge (L1000766 : y_5, y_4, r_4)

Validation par couleur : le `dominant_color` du crop sauvé doit correspondre à la couleur attendue. **Résultat : 54/54, 0 mismatch.**

In [ ]:
# Mosaïque des 54 templates extraits
img = cv2.imread(str(ROOT / 'outputs' / 'templates_preview.png'))
plt.figure(figsize=(16, 10))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title('54 card templates extracted from reference_images')
plt.tight_layout()
plt.show()

## 3. Augmentation synthétique

À partir de chaque template, on génère à la volée des centaines de variantes pendant l'entraînement. La pipeline d'augmentation simule **les imperfections de la détection en aval** plutôt que reconstruire des scènes complètes :

1. **Rotation discrète** {0°, 90°, 180°, 270°} — les cartes peuvent être posées dans n'importe quelle orientation
2. **Tilt fin** −15° à +15°
3. **Distorsion de perspective** (jitter ±12 px sur les coins) — vue de biais
4. **Échelle + translation** modérées
5. **Photométrie** : luminosité ±50, contraste ×0.65–1.35, saturation ×0.55–1.45, hue ±8°
6. **Flou gaussien** σ ≤ 2.0 et **bruit gaussien** σ ≤ 14
7. **BG fringe** (60 % des cas) : la carte est rétrécie à 82–98 % et placée sur un fond synthétique (blanc nuancé OU disques colorés bruités) pour simuler les leakages de fond observés sur les crops réels

Le BG fringe est crucial car les warps de détection capturent souvent un peu de fond aux bords, et un classifieur entraîné sans cette variation overfit aux templates parfaits.

In [ ]:
img = cv2.imread(str(ROOT / 'outputs' / 'augmentations_preview.png'))
plt.figure(figsize=(16, 14))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title('Augmentation : 1 original + 9 variantes par carte')
plt.tight_layout()
plt.show()

## 4. Architecture du CNN

**`UnoCNN`** : ResNet-18-light custom, **11.2M paramètres** (sous le plafond de 12M).

```
Stem  : Conv 3×3, 3 → 64 channels (pas de downsampling pour préserver les détails)
Stage 1 : 2 BasicBlocks 64
Stage 2 : 2 BasicBlocks 128 (downsample stride 2)
Stage 3 : 2 BasicBlocks 256 (downsample stride 2)
Stage 4 : 2 BasicBlocks 512 (downsample stride 2)
Head  : GAP + Dropout 0.3 + Linear → 54
```

Les **BasicBlocks** (Conv 3×3 + BN + ReLU + Conv 3×3 + BN + skip) facilitent l'optimisation et limitent le gradient vanishing — utile pour distinguer des classes visuellement proches (ex : `y_8` vs `y_9`, `b_skip` vs `b_reverse`).

Pourquoi pas plus gros ? Le règlement plafonne à 12M. Pourquoi pas plus petit ? Avec 27000 samples synthétiques par epoch et de l'augmentation forte, un modèle plus expressif tire mieux parti de la diversité.

**Input** : 144×96 RGB (rapport 2:3 du portrait, downscalé depuis 200×300 pour réduire la charge CPU au data loading). **Output** : 54 logits.

In [ ]:
import torch
from src.model import UnoCNN, count_parameters

model = UnoCNN()
print(f'Total parameters : {count_parameters(model):,}')
print(f'Under 12M limit  : {count_parameters(model) < 12_000_000}')

x = torch.randn(2, 3, 144, 96)
y = model(x)
print(f'Forward pass : input {tuple(x.shape)} -> output {tuple(y.shape)}')

## 5. Détection des cartes dans les scènes de jeu

Deux types de fonds nécessitent deux stratégies :

**Fond blanc** — saturation HSV directe :
- Cartes = blobs saturés isolés. Une gaussian blur sur la saturation propage les cadres colorés vers l'intérieur de chaque carte → blob plein par carte
- Pour les cartes mergées (overlapping ou L-shape), split géométrique le long du grand axe avec n estimé depuis la médiane des aires des cartes simples

**Fond bruité (feuillage)** — détection par les ovales blancs :
- La saturation est saturée par le feuillage → useless
- Mais les **ovales blancs** au centre de chaque carte forment des « trous enclosed » dans la saturation. Un floodfill depuis les coins identifie ces ovales
- Le rect de la carte est inféré en agrandissant l'ovale (×1.55)

**Filtrage des faux positifs** : chaque crop warpé doit avoir une signature de carte UNO (zone centrale blanche OU sombre OU bordure très saturée + intérieur clair).

In [ ]:
# Visualisation d'une détection sur image fond blanc et fond bruité
from src.detection import detect_cards_in_scene, detect_active_token

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, image_id, title in [(axes[0], 'L1000770', 'Fond blanc'), (axes[1], 'L1000909', 'Fond bruité')]:
    img = cv2.imread(str(TRAIN_DIR / f'{image_id}.jpg'))
    cards = detect_cards_in_scene(img)
    vis = img.copy()
    for c in cards:
        box = cv2.boxPoints(c['rect']).astype(np.intp)
        cv2.drawContours(vis, [box], 0, (0, 255, 0), 8)
    token = detect_active_token(img)
    if token:
        cv2.circle(vis, (int(token[0]), int(token[1])), 60, (255, 100, 0), 10)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{title} — {image_id} : {len(cards)} cartes détectées')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Détection du jeton actif

Le jeton vient en **2 formes** selon le fond :
- **Noir rectangulaire** sur fond blanc
- **Jaune rond** sur fond bruité

Détection adaptative basée sur la médiane V de l'image :
- Médiane V > 180 (fond blanc) → on cherche un blob sombre rectangulaire (V < median−80, aspect < 3, solidité > 0.85)
- Sinon → on cherche un disque jaune (HSV in-range pour le jaune, circularité > 0.7)

Le jeton est ensuite assigné au joueur dont la **carte la plus proche** est la sienne (plus robuste qu'une assignation purement angulaire pour les jetons dans les coins ambigus).


## 7. Pipeline d'inférence end-to-end

Pour chaque image :

```
image → detect_cards_in_scene() → liste de crops 200×300
      → classifier.classify(crops) → labels + confidences
      → filtrage par seuil de confiance
      → assign_player(cx,cy) pour chaque carte (center / p1-4)
      → detect_active_token() + assign_token_to_player()
      → ScenePrediction(center_card, active_player, player_cards)
      → CSV row
```

Le filtrage par confiance (`>= 0.40`) sert à écarter les détections faiblement classifiées (typiquement faux positifs sur fond bruité).

## 8. Résultats sur les 81 images train annotées

Métrique de la compétition : `Score = 0.1·CenterAcc + 0.1·ActiveAcc + 0.8·F1` où F1 est calculé en multi-ensemble sur les cartes des 4 joueurs.

| Itération | CenterAcc | ActiveAcc | F1 | Score |
|---|---|---|---|---|
| v1 (training initial) | 0.519 | 0.272 | 0.379 | 0.382 |
| v2 (+ token detection adaptatif) | 0.519 | 0.667 | 0.379 | 0.422 |
| v3 (+ filtre signature carte) | _à mesurer_ | _à mesurer_ | _à mesurer_ | _à mesurer_ |
| v4 (+ retrain avec bg fringe) | _à mesurer_ | _à mesurer_ | _à mesurer_ | _à mesurer_ |

*(table à compléter au fil des itérations)*

## 9. Cas d'échec et analyse

*(Section à compléter — visualisation des images où le pipeline échoue le plus, avec analyse causale par catégorie : sur-détection sur fond bruité, mauvaise classification, jeton non trouvé, etc.)*

## 10. Améliorations possibles

- **Détection sur fond bruité** : la signature actuelle laisse passer ~30 % de faux positifs. Une vérification CNN dédiée (binary classifier carte/non-carte) supprimerait ce bruit
- **Classifier** : entraîner avec une supervision plus directe — extraire des crops réels depuis les `train_images` annotées pour avoir des paires (crop, label) qui collent à ce que verra le classifieur en inférence
- **Cartes en chevauchement** : actuellement on segmente puis on split géométriquement. Un watershed sur la transformée de distance ferait mieux pour les chevauchements complexes
- **Active player** : au-delà du jeton, on pourrait combiner avec la position relative aux cartes (le joueur actif a souvent plus de cartes ou des cartes posées différemment)
